## Quality control per cluster, and add platform adjusted ic_id_sample

Library

In [1]:
# Path-related libraries
import os
from pyhere import here  # For reproducible relative paths
import sys # system specific parameters
from pathlib import Path # File system paths

# AnnData and single-cell analysis libraries
import scanpy as sc       # For preprocessing and visualization of single-cell data
import anndata as ad      # For handling AnnData objects

# Numerical operations
import numpy as np        # For numerical computations and array manipulations
import pandas as pd

# Custom modules and functions
sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import my_anndata as ma                    # Custom AnnData utilities

Parameters

In [2]:
# Path to save Anndata object
anndata_dir = str(here('data/anndata/'))

Load data

In [8]:
# A data object
adata = ad.read_h5ad(os.path.join(anndata_dir, 'AB_combined.h5ad'))
# Cells that failed QC
cells_keep = pd.read_csv(here('data/quality_control_per_cluster/first_pass/files/barcodes_pass_qc.csv'))

Keep only cells that passed per cluster quality control

In [9]:
mask = adata.obs.index.isin(cells_keep['barcode'])
adata_sub = adata[mask]

Add platform adjusted ic_id_sample

In [10]:
# you wont need to do this when you use AC_combined - which you will add 
# Extract old meta data
old_meta = adata_sub.obs.copy()

# Rename
adata_sub.obs.rename(columns={'ic_id_study': 'ic_id_dataset'}, inplace=True) # Current id reflects dataset not study
adata_sub.obs.rename(columns={'ic_id_study_overall': 'ic_id_study'}, inplace=True) # Now overall study can be called study
adata_sub.obs.rename(columns={'ic_id_donor': 'ic_id_dataset_donor'}, inplace=True) # Now id reflects donor per dataset

# Check renaming was done correctly
assert (old_meta['ic_id_study'] == adata_sub.obs['ic_id_dataset']).all()
assert (old_meta['ic_id_study_overall'] == adata_sub.obs['ic_id_study']).all()
assert (old_meta['ic_id_donor'] == adata_sub.obs['ic_id_dataset_donor']).all()

# Add platform adjusted id 
adata_sub.obs['ic_id_platform_adjusted_sample'] = np.where(
    adata_sub.obs['platform'].isin(['droplet', 'plate_barcode']),
    adata_sub.obs['ic_id_sample'],
    np.where(adata_sub.obs['platform'].isin(['plate']),
             adata_sub.obs['ic_id_dataset_donor'],
             None)  # fallback if platform is something else
)

/tmp/ipykernel_17149/1709775459.py:15: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_sub.obs['ic_id_platform_adjusted_sample'] = np.where(
